In [48]:
from __future__ import print_function, absolute_import
from gm.api import *
from gm.model.storage import context
from gm.model import DictLikeAlgoOrder
from gm.pb.account_pb2 import AlgoOrder
import numpy as np
import pandas as pd
from datetime import  datetime,timezone, timedelta
import datetime
import copy
import plotly.express as px
import plotly.graph_objects as go
pd.set_option('display.max_rows', None) #show all rows
pd.set_option('display.float_format', '{:.2f}'.format)
import warnings
warnings.filterwarnings('ignore')

#掘金量化
set_token('807b8ba88782d7343bfe3ad918f33f93a2610ee8')
account_id='28edb8c3-ed6d-11f0-9ac0-00163e022aa6'

In [ ]:
#KDJ
stock='SHSE.601606'
start='2025-08-25'
end='2026-02-13'
n=9

In [50]:
history_data = history(symbol=stock, frequency='1d', start_time=start,  end_time=end, fields='close,eob', adjust=ADJUST_PREV, df= True)
history_data['eob'] = history_data['eob'].dt.date  #优化日期格式

In [ ]:
history_data['max_9']=[50]*len(history_data)
history_data['min_9']=[50]*len(history_data)

for i in range(8,len(history_data)):
    max_i=max(history_data['close'][i-8:i+1])
    min_i=min(history_data['close'][i-8:i+1])

    history_data['max_9'][i]=max_i
    history_data['min_9'][i]=min_i
history_data

,close,eob,max_9,min_9
0,62.14,2025-08-25,50.00,50.00
1,65.45,2025-08-26,50.00,50.00
2,63.27,2025-08-27,50.00,50.00
3,62.88,2025-08-28,50.00,50.00
4,69.17,2025-08-29,50.00,50.00
5,70.70,2025-09-01,50.00,50.00
6,72.42,2025-09-02,50.00,50.00
7,65.18,2025-09-03,50.00,50.00
8,58.66,2025-09-04,72.42,58.66
9,53.55,2025-09-05,72.42,53.55


In [52]:
history_data['rsv']=[0]*len(history_data)
for i in range(8,len(history_data)):
    history_data['rsv'][i]=((history_data['close'][i]-history_data['min_9'][i]) /(history_data['max_9'][i]-history_data['min_9'][i])) * 100
history_data

,close,eob,max_9,min_9,rsv
0,62.14,2025-08-25,50.00,50.00,0.00
1,65.45,2025-08-26,50.00,50.00,0.00
2,63.27,2025-08-27,50.00,50.00,0.00
3,62.88,2025-08-28,50.00,50.00,0.00
4,69.17,2025-08-29,50.00,50.00,0.00
5,70.70,2025-09-01,50.00,50.00,0.00
6,72.42,2025-09-02,50.00,50.00,0.00
7,65.18,2025-09-03,50.00,50.00,0.00
8,58.66,2025-09-04,72.42,58.66,0.00
9,53.55,2025-09-05,72.42,53.55,0.00


In [53]:
history_data['k']=[0]*len(history_data)
for i in range(9,len(history_data)):
    history_data['k'][i]=(1/3)*history_data['rsv'][i] + (2/3)*history_data['k'][i-1]
history_data['d']=[0]*len(history_data)
for i in range(9,len(history_data)):
    history_data['d'][i]=(1/3)*history_data['k'][i] + (2/3)*history_data['d'][i-1]
history_data['j']=[0]*len(history_data)
for i in range(9,len(history_data)):
    history_data['j'][i]=3*history_data['k'][i] - 2*history_data['d'][i]
history_data

,close,eob,max_9,min_9,rsv,k,d,j
0,62.14,2025-08-25,50.00,50.00,0.00,0.00,0.00,0.00
1,65.45,2025-08-26,50.00,50.00,0.00,0.00,0.00,0.00
2,63.27,2025-08-27,50.00,50.00,0.00,0.00,0.00,0.00
3,62.88,2025-08-28,50.00,50.00,0.00,0.00,0.00,0.00
4,69.17,2025-08-29,50.00,50.00,0.00,0.00,0.00,0.00
5,70.70,2025-09-01,50.00,50.00,0.00,0.00,0.00,0.00
6,72.42,2025-09-02,50.00,50.00,0.00,0.00,0.00,0.00
7,65.18,2025-09-03,50.00,50.00,0.00,0.00,0.00,0.00
8,58.66,2025-09-04,72.42,58.66,0.00,0.00,0.00,0.00
9,53.55,2025-09-05,72.42,53.55,0.00,0.00,0.00,0.00


In [ ]:
fig = px.line(history_data, x='eob', y=['k', 'd','j'],color_discrete_sequence=['white','yellow','purple'])  
fig.show()